In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm

%matplotlib inline
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"delivery_time Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
df = df.drop('Order_ID', axis=1)

In [ ]:
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

# check_missing_values(df)
# numerical_cols =  df.select_dtypes(include=["float64","int"]).columns
# print(numerical_cols.shape)
# # df_clean = df[cols_N].copy()
# # cols_S=['Weather','Traffic_Level', 'Courier_Experience_yrs' ]
# # # Fill missing values with mean for each column
# # df = df.fillna(df[cols_N].mean())
# # df=df[cols_S].fillna('non')

In [ ]:
# Select relevant columns, we do not select 'model' (too many unique values, too sparse and will hurt the model performance)
cols=[ 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs',
       'Delivery_Time']
df_clean = df[cols].copy()


# Drop rows where target or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time','Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs'])
print(f"After dropping missing price/year/odometer: {df_clean.shape}")


In [ ]:
df_clean.info()

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
df_clean.head()

In [ ]:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
print(df_clean.columns)

In [ ]:
# Task 5: Write your code here:
features = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop('Delivery_Time', axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

In [ ]:
RF=RandomForestRegressor(n_estimators=200)

In [ ]:
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200)
}

In [ ]:
# Task 2,3,4,5: Write your code here:



# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}


n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)

In [ ]:
# Task 1: Write your code here:
coeffs = {}



fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: